In [1]:
# ============================================================
# Cell 1: ENVIRONMENT GATE — Colab A100 ONLY
# ============================================================
# MANDATORY: This cell must pass before ANY training.
# Refuses local execution. Verifies CUDA GPU.
import os, sys

_cwd = os.getcwd()
_is_local = _cwd.startswith("/Users/") or (_cwd.startswith("/home/") and "content" not in _cwd)

import torch
_has_cuda = torch.cuda.is_available()

if _is_local or not _has_cuda:
    print("=" * 65)
    print("  BLOCKED: This notebook must run on Google Colab with CUDA GPU")
    print(f"  Current dir : {_cwd}")
    print(f"  CUDA        : {_has_cuda}")
    print("=" * 65)
    print("\n  Runtime → Change runtime type → A100 GPU")
    raise SystemExit("Refusing local execution. Use Colab A100.")

# Passed — CUDA environment verified
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)

print(f"{'='*65}")
print(f"  ENVIRONMENT VERIFIED")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram_gb:.1f} GB")
print(f"  Compute cap  : {cc[0]}.{cc[1]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")
print(f"  Working dir  : {_cwd}")
print(f"{'='*65}")
os.system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

  ENVIRONMENT VERIFIED
  GPU          : NVIDIA A100-SXM4-40GB
  VRAM         : 42.4 GB
  Compute cap  : 8.0
  PyTorch      : 2.10.0+cu128
  CUDA         : 12.8
  Working dir  : /content


0

In [51]:
# ============================================================
# Cell 2: Clone Repo + Install Dependencies + Build Wiki Cache
# ============================================================
import subprocess, os, sys, pathlib

REPO_URL = "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git"
PROJ_ROOT = "/content/nst"

if not os.path.isdir(os.path.join(PROJ_ROOT, "data")):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, PROJ_ROOT], check=True)
else:
    # Pull latest changes
    subprocess.run(["git", "pull", "--ff-only"], cwd=PROJ_ROOT, check=True)

os.chdir(PROJ_ROOT)
sys.path.insert(0, PROJ_ROOT)

# Pin datasets<3 (datasets 3.x dropped trust_remote_code, breaks FEVER loading)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets>=2.18.0,<3.0.0"], check=True)

# Install requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."],
               cwd=PROJ_ROOT, check=True)

# Import datasets EARLY (before torch) to register pyarrow extension types cleanly
import datasets
print(f"datasets version: {datasets.__version__}")

print(f"\nProject root: {PROJ_ROOT}")
print(f"Python: {sys.executable}")

# ── Build FEVER wiki cache if not present ─────────────────
wiki_db = os.path.join(PROJ_ROOT, "data", "fever_wiki.db")
if not os.path.exists(wiki_db):
    print("\n>>> Building FEVER wiki cache (one-time, ~5 min)...")
    subprocess.run([sys.executable, "main.py", "build-fever-wiki-cache"],
                   cwd=PROJ_ROOT, check=True)
    print(f"  Wiki cache built: {wiki_db}")
else:
    sz = os.path.getsize(wiki_db) / (1024*1024)
    print(f"\n  Wiki cache exists: {wiki_db} ({sz:.1f} MB)")

# GPU auto-config for A100
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1e9
    cc = (props.major, props.minor)

    # A100/H100 optimizations
    if cc >= (8, 0):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    bs = 32 if vram > 30 else (16 if vram > 12 else 8)
    ga = 2 if bs == 32 else 4
    print(f"\nGPU config: bs={bs}×{ga}={bs*ga} effective, BF16, VRAM={vram:.0f}GB")

datasets version: 2.21.0

Project root: /content/nst
Python: /usr/bin/python3

  Wiki cache exists: /content/nst/data/fever_wiki.db (24.2 MB)

GPU config: bs=32×2=64 effective, BF16, VRAM=42GB


In [ ]:
# Test: model can learn WITHOUT LoRA? And check LoRA target_modules
import subprocess
r = subprocess.run(["python", "-c", """
import sys; sys.path.insert(0, '/content/nst')
import os; os.chdir('/content/nst')
import torch
import torch.nn.functional as F
from functools import partial
from data.fever_dataset import load_fever_splits, FeverGoldDataset, fever_collate_fn
from transformers import AutoModelForSequenceClassification, AutoConfig, DebertaV2Tokenizer

# Build model WITHOUT LoRA (full fine-tuning)
model_name = 'microsoft/deberta-v3-large'
tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name, num_labels=3)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config, torch_dtype=torch.float32)
model = model.cuda()

# Only finetune classifier + last 2 layers
for p in model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True
for p in model.pooler.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable (classifier+pooler only): {trainable/1e6:.2f}M")

splits = load_fever_splits(max_train=64, max_dev=10, dev_test_ratio=0.1, seed=42)
ds = FeverGoldDataset(splits['train'][:32])
collate = partial(fever_collate_fn, tokenizer=tokenizer, max_length=384)
from torch.utils.data import DataLoader
dl = DataLoader(ds, batch_size=32, collate_fn=collate)
batch = next(iter(dl))
labels = batch['labels']
print(f"Labels: {torch.bincount(labels, minlength=3).tolist()}")

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
model.train()
for step in range(50):
    optimizer.zero_grad()
    out = model(input_ids=batch['input_ids'].cuda(), attention_mask=batch['attention_mask'].cuda())
    logits = out.logits
    loss = F.cross_entropy(logits, labels.cuda())
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        preds = logits.argmax(-1)
        acc = (preds == labels.cuda()).float().mean()
        print(f"Step {step}: loss={loss.item():.4f}, acc={acc.item():.4f}, preds={torch.bincount(preds, minlength=3).tolist()}")

print()
print("--- Now test with LoRA ---")
# Check: what modules does LoRA actually target?
from peft import LoraConfig, get_peft_model, TaskType
model2 = AutoModelForSequenceClassification.from_pretrained(model_name, config=config, torch_dtype=torch.float32)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["query_proj", "key_proj", "value_proj", "dense"],
    bias="none", modules_to_save=["classifier", "pooler"],
)
model2 = get_peft_model(model2, lora_config)
# Check which modules actually got LoRA
lora_modules = [n for n, _ in model2.named_modules() if 'lora' in n.lower()]
print(f"LoRA modules found: {len(lora_modules)}")
if len(lora_modules) < 5:
    print("WARNING: Very few LoRA modules!")
    for m in lora_modules[:20]:
        print(f"  {m}")
else:
    print(f"First 5: {lora_modules[:5]}")
"""], capture_output=True, text=True, cwd="/content/nst")
print(r.stdout[:3000])
if r.returncode != 0:
    print("ERR:", r.stderr[-1500:])

In [52]:
# ============================================================
# Cell 3: 10K NEURAL BASELINE (subprocess — avoids import conflicts)
# ============================================================
import subprocess, sys, json, os, shutil, time

# Clean previous run
outdir = "/content/nst/outputs_neural_10k"
if os.path.isdir(outdir):
    shutil.rmtree(outdir)
    print(f"Cleaned previous: {outdir}")

print("=" * 65)
print("  10K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 5")
print("  A100: bs=32×2=64 effective, FP32+TF32")
print("=" * 65)

t0 = time.time()
ret = subprocess.run(
    [sys.executable, "main.py", "train-fever-nst",
     "--config", "configs/fever_neural_10k_a100.yaml",
     "--outdir", "outputs_neural_10k"],
    cwd="/content/nst"
)
elapsed = time.time() - t0
print(f"\nTraining took {elapsed/60:.1f} min")

if ret.returncode != 0:
    raise RuntimeError(f"Neural baseline failed with code {ret.returncode}")

# Load saved report
with open("/content/nst/outputs_neural_10k/report.json") as f:
    results_neural = json.load(f)

with open("/content/nst/results_neural_10k.json", "w") as f:
    json.dump(results_neural, f, indent=2)

print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 10K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
dev = results_neural.get("dev", {})
print(f"  Dev acc    : {dev.get('accuracy', '?')}")
print(f"  Dev ECE    : {dev.get('ece', '?')}")
print(f"  Best dev   : {results_neural.get('best_dev_acc', '?')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats['accuracy']:.4f} (n={stats['count']})")
dt = results_neural.get("dev_test", {})
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")
print(f"\n  Saved to results_neural_10k.json")

Cleaned previous: /content/nst/outputs_neural_10k
  10K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)
  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 5
  A100: bs=32×2=64 effective, FP32+TF32

Training took 30.9 min

  NEURAL BASELINE 10K RESULTS (30.9 min)
  Dev acc    : 0.3326
  Dev ECE    : 0.005543
  Best dev   : 0.3493
    SUPPORTS            : 1.0000 (n=898)
    REFUTES             : 0.0000 (n=870)
    NOT ENOUGH INFO     : 0.0000 (n=932)

  Held-out dev_test accuracy: 0.32

  Saved to results_neural_10k.json


In [ ]:
# ============================================================
# Cell 4: 10K NST-VERI v0.4 (subprocess — avoids import conflicts)
# ============================================================
import subprocess, sys, json, os

print("=" * 65)
print("  10K NST-VERI v0.4: evidence-gated + inference-time fusion")
print("  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 8")
print("  Changes: evidence gating, simplified constraint loss,")
print("           inference-time constraint fusion, λ_max=0.5, β_aux=0.3")
print("  A100 optimized: bs=32×2=64 effective, BF16, TF32, focal loss")
print("=" * 65)

ret = subprocess.run(
    [sys.executable, "main.py", "train-fever-veri",
     "--config", "configs/fever_veri_10k_a100.yaml",
     "--outdir", "outputs_veri_10k"],
    cwd="/content/nst"
)
if ret.returncode != 0:
    raise RuntimeError(f"NST-VERI failed with code {ret.returncode}")

# Load saved report
with open("/content/nst/outputs_veri_10k/report.json") as f:
    results_veri = json.load(f)

# Also save a copy at project root
with open("/content/nst/results_veri_10k.json", "w") as f:
    json.dump(results_veri, f, indent=2)

print(f"\n{'='*65}")
print(f"  NST-VERI 10K RESULTS ({results_veri.get('elapsed_s', 0)/60:.1f} min)")
print(f"{'='*65}")
dev = results_veri.get("dev", {})
print(f"  Dev acc (fused): {dev.get('accuracy', '?')}")
print(f"  Dev ECE        : {dev.get('ece', '?')}")
print(f"  Best dev (raw) : {results_veri.get('best_dev_acc', '?')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats['accuracy']:.4f} (n={stats['count']})")
dt = results_veri.get("dev_test", {})
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")
print(f"\n  Saved to results_veri_10k.json")

In [ ]:
# ============================================================
# Cell 5: 10K COMPARISON — Neural vs NST-VERI v0.4 (HONEST REPORT)
# ============================================================
import json, os

experiments = {}
for name, path in [("neural_10k", "results_neural_10k.json"),
                   ("veri_10k", "results_veri_10k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*70}")
print(f"  10K COMPARISON: Neural vs NST-VERI v0.4 (FEVER Gold Evidence)")
print(f"{'='*70}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8} {'DevTest':>10}")
print(f"  {'─'*63}")

for name, r in experiments.items():
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "?")
    best = r.get("best_dev_acc", "?")
    ece = dev.get("ece", "?")
    dt = r.get("dev_test", {})
    dt_acc = dt.get("accuracy", "?") if dt else "?"
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
    print(f"  {name:<25} {fmt(acc):>10} {fmt(best):>10} {fmt(ece):>8} {fmt(dt_acc):>10}")

# Per-label breakdown
for name, r in experiments.items():
    dev = r.get("dev", {})
    print(f"\n  {name} per-label:")
    for label, stats in dev.get("per_label", {}).items():
        print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Delta
if "neural_10k" in experiments and "veri_10k" in experiments:
    n_dev = experiments["neural_10k"].get("dev", {}).get("accuracy", 0)
    v_dev = experiments["veri_10k"].get("dev", {}).get("accuracy", 0)
    n_held = experiments["neural_10k"].get("dev_test", {}).get("accuracy", 0) if experiments["neural_10k"].get("dev_test") else 0
    v_held = experiments["veri_10k"].get("dev_test", {}).get("accuracy", 0) if experiments["veri_10k"].get("dev_test") else 0
    n_best = experiments["neural_10k"].get("best_dev_acc", 0)
    v_best = experiments["veri_10k"].get("best_dev_acc", 0)
    
    print(f"\n{'='*70}")
    print(f"  DELTAS (VERI - Neural):")
    print(f"    Dev accuracy:     {v_dev - n_dev:+.4f}")
    print(f"    Best dev:         {v_best - n_best:+.4f}")
    print(f"    Held-out:         {v_held - n_held:+.4f}")
    
    if v_dev > n_dev and v_held > n_held:
        print(f"\n  ✅ NST-VERI v0.4 SUPERIOR on both dev and held-out")
    elif v_dev > n_dev or v_held > n_held:
        print(f"\n  ⚠️ NST-VERI v0.4 wins on one metric, mixed on the other")
    else:
        print(f"\n  ❌ Neural baseline still ahead — need further improvements")
    
    print(f"\n  INTEGRITY:")
    print(f"    Train/dev split: separate (dev_test_ratio=0.1 held out)")
    print(f"    Same backbone: DeBERTa-v3-large + LoRA for both")
    print(f"    Same data: 12k train / 3k dev for both")
    print(f"    Same seed: 42")
    print(f"    Evaluation: dev set only, NOT on train set")
    print(f"    VERI uses inference-time constraint fusion (honest advantage)")
print(f"{'='*70}")